# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m02_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 시계열 비행 궤적 데이터 정제 및 이상 조작 노이즈 필터링 전처리 구현
**소요시간**: 약 2.0시간 | **필수 산출물**: src/preprocess/trn01_preprocessor.py (필수), 전처리 전후 데이터셋 (필수)

### 📋 미션 개요 (Overview)
원시 비행로그에는 시각 중복·역행, 센서 단절, GPS 튐, 원시 PWM 스틱 신호가 섞여 있어 그대로 채점하면 숙련도가 왜곡된다. 본 미션에서는 10Hz 격자 정렬, 스틱 PWM(1000~2000us)의 -1~1 정규화 및 데드밴드 적용, Hampel 필터 기반 GPS 이상치 제거와 단절 구간 보간을 수행하는 전처리 모듈(`trn01_preprocessor.py`)을 구현한다. 주간·야간·돌풍 조건별 임계값 차이를 비교하고 처리 속도(1,000건/초 이상)와 단위 테스트 5종 통과 여부를 검증한다.

### 🎓 학습 목표 (Objectives)
불규칙한 센서 시계열에서 **물리적·전술적 의미를 훼손하지 않고 잡음과 결측을 제거하는 신호 전처리 원리**를 익힌다. 10Hz 리샘플링과 스틱 PWM(1000~2000us) 정규화·데드밴드 처리, Hampel 필터를 이용한 GPS 튐 제거 및 단절 구간 보간 기법을 학습하며, 주간·야간·돌풍 조건별로 이상치 임계값을 차등 설계해 실제 급기동 보존과 노이즈 억제 사이의 트레이드오프를 판단하는 감각을 기른다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션2 코드를 실행해 처리 속도와 결측 보정율을 확인한다.
#### 이 단계의 역할
미션2 코드를 실행해 처리 속도와 결측 보정율을 확인한다.

#### 해야 할 일

- 미션2 코드를 실행해 처리 속도와 결측 보정율을 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 처리 속도 1,000건/초 이상(예시 약 9만 레코드/초), 오류 0건 결측 보정율 100%, 단위 테스트 5/5 통과

#### 학습자가 제출해야 하는 내용

- 미션2 코드를 실행해 처리 속도와 결측 보정율을 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 비행로그 시계열을 10Hz(0.1초) 격자로 정렬하면서 시각 중복·역행을 제거하고, 스틱 PWM(1000~2000us)을 -1~1로 정규화(중립 데드밴드 포함)하는 벡터화 Python 함수를 작성해줘.
**AI 응답 검증 포인트**

- 처리 속도가 기준치(1,000건/초 이상)를 충족하는지 실제 실행 시간(records/sec)으로 검증
- 중립(1500us 부근) 데드밴드 구간이 정확히 0으로 매핑되고 1000~2000us 경계값이 -1, +1로 변환되는지 확인

#### Step 2. 조건별 전처리 효과표에서 위치 오차가 얼마나 줄었는지 비교한다.
#### 이 단계의 역할
조건별 전처리 효과표에서 위치 오차가 얼마나 줄었는지 비교한다.

#### 해야 할 일

- 조건별 전처리 효과표에서 위치 오차가 얼마나 줄었는지 비교한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 처리 속도 1,000건/초 이상(예시 약 9만 레코드/초), 오류 0건 결측 보정율 100%, 단위 테스트 5/5 통과

#### 학습자가 제출해야 하는 내용

- 조건별 전처리 효과표에서 위치 오차가 얼마나 줄었는지 비교한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 주간·야간·돌풍 조건별로 Hampel 필터 임계값(K)을 다르게 적용해 GPS 튐을 제거하고, 조건별 전처리 전후 위치 RMSE 개선율을 비교표로 출력하는 코드를 작성해줘.
**AI 응답 검증 포인트**

- 조건별 개선 수치가 실제 데이터셋 연산 결과인지(AI가 임의로 추정한 수치가 아닌지) 로그로 확인
- 단절 구간 선형/스플라인 보간 후 NaN 잔여 값이 0건인지 검증

#### Step 3. 전후 비교 그림에서 튐 값과 단절 구간이 처리된 모습을 확인한다.
#### 이 단계의 역할
전후 비교 그림에서 튐 값과 단절 구간이 처리된 모습을 확인한다.

#### 해야 할 일

- 전후 비교 그림에서 튐 값과 단절 구간이 처리된 모습을 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 처리 속도 1,000건/초 이상(예시 약 9만 레코드/초), 오류 0건 결측 보정율 100%, 단위 테스트 5/5 통과

#### 학습자가 제출해야 하는 내용

- 전후 비교 그림에서 튐 값과 단절 구간이 처리된 모습을 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 전처리 전후의 GPS 경로(x, y)와 스틱 입력 시계열을 나란히 시각화하여, 튐 값(Spike)과 센서 단절 보간 구간이 한눈에 구분되도록 matplotlib 그래프를 그려줘.
**AI 응답 검증 포인트**

- 그래프에서 정상 급기동 구간까지 과도하게 평활화(과필터링)되지 않았는지 육안 점검
- 단절 구간이 직선 또는 자연스러운 궤적으로 복원됐는지 전후 좌표를 대조

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
HAMPEL_K의 gust 값을 3.0으로 낮추면 돌풍 조건의 이상치 수와 위치 오차가 어떻게 바뀌는지 확인하고, 실제 급기동이 지워질 위험을 토의한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 돌풍 조건의 HAMPEL_K 값을 3.0으로 낮췄을 때 이상치 탐지 건수와 위치 오차가 어떻게 변하는지 비교 실험 코드를 짜고, 실제 급기동이 노이즈로 오인될 위험을 분석해줘.
**AI 응답 검증 포인트**

- K 값 변경 전후의 이상치 수·위치 오차 변화가 실제 실행 수치와 일치하는지 확인
- 돌풍 회피를 위한 정상 급조작이 지워지는 False Positive 사례가 있는지 원본 로그와 대조


In [ ]:
"""시뮬레이터 원시 비행로그의 시각 중복·역행 정리, 스틱 PWM 값(1000~2000us) 정규화, GPS 튐(이상치) 제거(Hampel 필터), 센서 단절 구간 보간을 수행하는 전처리 모듈을 구현하고, 비행 조건(주간/야간/돌풍)별 임계값을 달리 적용해 처리 속도와 위치 오차 개선을 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 드론 조종 시뮬레이터 비행로그 생성기 (실제 훈련생·장비 정보 없음, 조종자는 가명 P001~)
# 비행 1회 = 120초(10Hz): 경로 비행(0~60초) -> 호버링(60~85초) -> 돌발 장애물 회피(85초) -> 비상 착륙(100초 지시)
# 조종자마다 숨은 숙련도(skill)와 조작 습관(급조작 jerky, 과보정 overcorr)이 있고, 교관 점수는 이를 반영해 만든다.
# ---------------------------------------------------------------------------
DT, STEPS = 0.1, 1200
T = np.arange(STEPS) * DT
COND = {"day": dict(wind=0.4, gust=0.0, gps=0.25, delay=0.0), "night": dict(wind=0.4, gust=0.0, gps=0.5, delay=0.15),
        "gust": dict(wind=1.2, gust=2.5, gps=0.25, delay=0.0)}
OBST_T, LAND_T = 85.0, 100.0


def make_pilots(n=40, seed=7):
    rng = np.random.default_rng(seed)
    skill = rng.beta(2.2, 2.0, n)
    return pd.DataFrame({"pilot": [f"P{i + 1:03d}" for i in range(n)], "skill": skill,
                         "jerky": rng.random(n) < 0.15 + 0.35 * (1 - skill), "overcorr": rng.random(n) < 0.15 + 0.35 * (1 - skill)})


def target_path(t):
    """사각형 경로(40x30m, 고도 20m)를 3m/s로 비행 -> 60초 지점에서 호버링 -> 착륙 지시 후 고도 0."""
    # [TODO : 학생 실습 2] 시계열 비행 궤적 데이터 정제 및 이상 조작 노이즈 필터링 전처리 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def fly(pilots, cond="day", seed=0, delay_extra=0.0, obstacle_side=None):
    """여러 조종자의 비행 1회를 동시에 모의한다(벡터화). 반환: 로그 배열 dict, 비행별 정답 정보 DataFrame."""
    # [TODO : 학생 실습 2] 시계열 비행 궤적 데이터 정제 및 이상 조작 노이즈 필터링 전처리 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def one(log, j):
    """여러 조종자 로그에서 j번째 조종자 1명의 로그(DataFrame)를 꺼낸다."""
    return pd.DataFrame({k: (v if v.ndim == 1 else v[:, j]) for k, v in log.items()})
# ---------------------------------------------------------------------------

HAMPEL_K = {"day": 3.0, "night": 3.5, "gust": 4.5}           # 조건별 이상치 판정 임계값(돌풍은 실제 큰 움직임이 많아 느슨하게)


def make_raw(log, j, cond, rng):
    """[교육용가상자료] 원시 로그: PWM 스틱값, GPS 튐 1%, 센서 단절(0.5~2초) 2회, 시각 중복 2%."""
    d = one(log, j)
    raw = d[["t", "x", "y", "alt", "roll_deg", "pitch_deg", "true_x", "true_y", "true_alt"]].copy()
    for s in ("stick_roll", "stick_pitch", "throttle"):
        raw[s + "_pwm"] = (1500 + 500 * d[s] + rng.normal(0, 4, len(d))).round().astype(int)
    k = rng.random(len(raw)) < 0.01
    for c in ("x", "y"):
        raw.loc[k, c] += rng.choice([-1, 1], k.sum()) * rng.uniform(6, 15, k.sum())
    for _ in range(2):
        a = int(rng.integers(50, len(raw) - 30)); raw.loc[a:a + int(rng.integers(5, 20)), ["x", "y", "alt"]] = np.nan
    dup = raw.sample(frac=0.02, random_state=int(rng.integers(1e6)))
    raw = pd.concat([raw, dup]).sample(frac=1.0, random_state=1).reset_index(drop=True)     # 순서 섞임 + 중복
    raw["t"] = raw["t"] + rng.normal(0, 0.003, len(raw))                                     # 시각 흔들림
    return raw


def pwm_to_stick(p, dead=8):
    v = (np.asarray(p, float) - 1500) / 500
    v[np.abs(np.asarray(p) - 1500) < dead] = 0.0                                             # 중립 부근 데드밴드
    return np.clip(v, -1, 1)


def hampel(x, win=7, k=3.0):
    s = pd.Series(x)
    med = s.rolling(win, center=True, min_periods=1).median()
    mad = (s - med).abs().rolling(win, center=True, min_periods=1).median() * 1.4826
    out = (s - med).abs() > k * (mad + 0.3)          # 0.3m: GPS 잡음 수준 이하 흔들림은 이상치로 보지 않음
    return np.where(out, np.nan, s), int(out.sum())


def preprocess(raw, cond):
    d = raw.copy()
    d["k"] = (d["t"] / DT).round().astype(int)                                             # 10Hz 격자 번호로 시각 정렬
    d = d.sort_values("k").drop_duplicates("k").set_index("k")
    d = d.reindex(np.arange(0, d.index.max() + 1))
    d["t"] = np.round(d.index * DT, 1)
    n_missing = int(d[["x", "y", "alt"]].isna().any(axis=1).sum())
    n_out = 0
    for c in ("x", "y", "alt"):
        d[c], n = hampel(d[c].values, k=HAMPEL_K[cond]); n_out += n
    d[["x", "y", "alt"]] = d[["x", "y", "alt"]].interpolate(limit_direction="both")
    for s in ("stick_roll", "stick_pitch", "throttle"):
        d[s] = pwm_to_stick(d[s + "_pwm"].interpolate(limit_direction="both").round())
    return d.reset_index(drop=True), n_missing, n_out


pilots = make_pilots(30, seed=7)
rng = np.random.default_rng(3)
rows, n_rec, t_total = [], 0, 0.0
for cond in COND:
    log, _ = fly(pilots, cond, seed=1)
    for j in range(len(pilots)):
        raw = make_raw(log, j, cond, rng)
        t0 = time.perf_counter()
        clean, miss, out = preprocess(raw, cond)
        t_total += time.perf_counter() - t0; n_rec += len(raw)
        e0 = np.sqrt((raw["x"] - raw["true_x"]) ** 2 + (raw["y"] - raw["true_y"]) ** 2).mean()
        e1 = np.sqrt((clean["x"] - clean["true_x"]) ** 2 + (clean["y"] - clean["true_y"]) ** 2).mean()
        rows.append({"조건": cond, "결측행": miss, "이상치": out, "남은결측": int(clean[["x", "y", "alt"]].isna().sum().sum()),
                     "위치오차 전(m)": e0, "위치오차 후(m)": e1})
df = pd.DataFrame(rows)
print(f"원시 레코드 {n_rec:,}개(로그 {len(df)}개) 처리, {n_rec / t_total:,.0f} 레코드/초 (목표 1,000건/초 이상), 처리 중 오류 0건")
print(f"결측 보정율 {1 - df['남은결측'].sum() / max(df['결측행'].sum(), 1):.0%} (결측행 {df['결측행'].sum()}개 모두 보간)")
print("\n[조건별 전처리 효과(로그 평균)]")
print(df.groupby("조건")[["결측행", "이상치", "위치오차 전(m)", "위치오차 후(m)"]].mean().round(2).to_string())
print("조건별 Hampel 임계값:", HAMPEL_K, "-> 돌풍 조건은 실제 급기동을 이상치로 지우지 않도록 느슨하게 둔다")

# 단위 테스트 5종
raw = make_raw(fly(pilots.iloc[:1], "day", seed=9)[0], 0, "day", np.random.default_rng(0))
clean, _, _ = preprocess(raw, "day")
spike = np.r_[np.zeros(10), 50.0, np.zeros(10)]
tests = {
    "PWM 1000/1500/2000 -> -1/0/1": np.allclose(pwm_to_stick([1000, 1500, 2000]), [-1, 0, 1]),
    "데드밴드(1504 -> 0)": pwm_to_stick([1504])[0] == 0.0,
    "단일 튐 값 제거": np.isnan(hampel(spike)[0][10]),
    "시각 중복 제거·단조 증가": clean["t"].is_monotonic_increasing and clean["t"].duplicated().sum() == 0,
    "결측 없음 + 10Hz 1,200행": clean[["x", "y", "alt"]].isna().sum().sum() == 0 and len(clean) == STEPS,
}
for k, v in tests.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"단위 테스트 {sum(tests.values())}/{len(tests)} 통과")

seg = slice(400, 700)
r2 = raw.sort_values("t").iloc[seg]
plt.figure(figsize=(9, 3.5))
plt.plot(r2["t"], r2["y"], ".", ms=3, c="gray", label="raw (spikes, gaps)")
plt.plot(clean["t"].iloc[seg], clean["y"].iloc[seg], c="tab:blue", label="preprocessed")
plt.plot(clean["t"].iloc[seg], clean["true_y"].iloc[seg], "k:", label="true")
plt.xlabel("time (s)"); plt.ylabel("y (m)"); plt.legend(); plt.title("Flight log preprocessing (synthetic)")
plt.tight_layout(); plt.show()

# TODO: 전처리 모듈을 src/preprocess/trn01_preprocessor.py 로, 전후 샘플을 data/processed/ 에 저장하세요.


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
원시 레코드 110,160개(로그 90개) 처리, 94,969 레코드/초 (목표 1,000건/초 이상), 처리 중 오류 0건
결측 보정율 100% (결측행 2348개 모두 보간)

[조건별 전처리 효과(로그 평균)]
         결측행    이상치  위치오차 전(m)  위치오차 후(m)
조건                                       
day    25.37  24.63       0.47       0.31
gust   26.17  21.87       0.45       0.33
night  26.73  27.90       0.78       0.62
조건별 Hampel 임계값: {'day': 3.0, 'night': 3.5, 'gust': 4.5} -> 돌풍 조건은 실제 급기동을 이상치로 지우지 않도록 느슨하게 둔다
[PASS] PWM 1000/1500/2000 -> -1/0/1
[PASS] 데드밴드(1504 -> 0)
[PASS] 단일 튐 값 제거
[PASS] 시각 중복 제거·단조 증가
[PASS] 결측 없음 + 10Hz 1,200행
단위 테스트 5/5 통과
```
</details>
